In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Fetch the world-famous UCI Concrete Dataset directly from the cloud
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
print("--- Connecting to cloud server to fetch 1,030 lab mixes... ---")

# Read the Excel file sheet directly into a Pandas DataFrame
raw_df = pd.read_excel(url)

# Rename columns to clear engineering variables for easier layout mapping
concrete_df = raw_df.rename(columns={
    raw_df.columns[0]: 'Cement',
    raw_df.columns[1]: 'Blast_Furnace_Slag',
    raw_df.columns[2]: 'Fly_Ash',
    raw_df.columns[3]: 'Water',
    raw_df.columns[4]: 'Superplasticizer',
    raw_df.columns[5]: 'Coarse_Aggregate',
    raw_df.columns[6]: 'Fine_Aggregate',
    raw_df.columns[7]: 'Age_Days',
    raw_df.columns[8]: 'Compressive_Strength'
})

print(f"Dataset Loaded Successfully! Found {concrete_df.shape[0]} unique lab mixes.\n")

# 2. Extract Features (X) and Target (y)
X_large = concrete_df.drop(columns=['Compressive_Strength'])
y_large = concrete_df['Compressive_Strength']

# 3. Train/Test Split (80% training / 20% validation)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_large, y_large, test_size=0.2, random_state=42)

# 4. Deploy and train an expanded Forest (150 trees)
production_forest = RandomForestRegressor(n_estimators=150, random_state=42)
production_forest.fit(X_train_l, y_train_l)

# 5. Evaluate overall performance
large_preds = production_forest.predict(X_test_l)
large_mae = mean_absolute_error(y_test_l, large_preds)
r2 = r2_score(y_test_l, large_preds)

print("--- Production Model Evaluation Metrics ---")
print(f"Mean Absolute Error (MAE): {large_mae:.2f} MPa")
print(f"Model R-Squared Score (R²): {r2:.4f} (Accuracy rate of ~{r2*100:.1f}%)")


--- Connecting to cloud server to fetch 1,030 lab mixes... ---
Dataset Loaded Successfully! Found 1030 unique lab mixes.

--- Production Model Evaluation Metrics ---
Mean Absolute Error (MAE): 3.79 MPa
Model R-Squared Score (R²): 0.8796 (Accuracy rate of ~88.0%)


In [7]:
X_test_l
y_test_l


31     52.908320
109    55.895819
136    74.497882
88     35.301171
918    10.535193
         ...    
482    56.144031
545    18.746163
110    37.997022
514    74.364911
602    35.170171
Name: Compressive_Strength, Length: 206, dtype: float64

In [14]:
import sys
# Automatically download Gradio directly within your notebook shell space if it isn't set up yet
try:
    import gradio as gr
except ImportError:
    import sys
    !{sys.executable} -m pip install gradio
    import gradio as gr

# 1. Define the prediction execution route function for the web wrapper interface
def predict_concrete_strength(cement, slag, ash, water, plasticizer, coarse_agg, fine_agg, age):
    # Pack user slider inputs into an identical row vector array format matching the model features
    input_array = np.array([[cement, slag, age, water, plasticizer, coarse_agg, fine_agg, age]])
    
    # Map the exact naming matrix frame columns to ensure scikit-learn features align safely
    input_df = pd.DataFrame(input_array, columns=X_large.columns)
    
    # Run the user input array down our trained forest consensus nodes
    val_prediction = production_forest.predict(input_df)[0]
    
    # Return a clean output string format to present directly inside the user text display block
    return f"✨ Predicted Compressive Strength: {val_prediction:.2f} MPa"

# 2. Build the graphical visual block elements composition layout
app_interface = gr.Interface(
    fn=predict_concrete_strength,
    inputs=[
        gr.Slider(100, 550, value=300, label="Cement (kg/m³)"),
        gr.Slider(0, 350, value=0, label="Blast Furnace Slag (kg/m³)"),
        gr.Slider(0, 200, value=0, label="Fly Ash (kg/m³)"),
        gr.Slider(120, 250, value=180, label="Water (kg/m³)"),
        gr.Slider(0, 35, value=0, label="Superplasticizer (kg/m³)"),
        gr.Slider(700, 1200, value=1000, label="Coarse Aggregate (kg/m³)"),
        gr.Slider(500, 1000, value=800, label="Fine Aggregate (kg/m³)"),
        gr.Slider(1, 365, value=28, label="Curing Age (Days)")
    ],
    outputs=gr.Textbox(label="AI Strength Assessment Indicator Result"),
    title="🏢 AI Concrete Compressive Strength Predictor",
    description="Slide the material formulation values to evaluate real-time physical performance predictions calculated across 1,000+ historical material core breaks."
)

# 3. Open a portal tunnel and spawn the live functional app dashboard
app_interface.launch(share=True , theme=gr.themes.Citrus())

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://860e84c959ee79e595.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using existing dataset file at: .gradio\flagged\dataset1.csv


In [2]:
import sys
!{sys.executable} -m pip install xlrd


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
